# Clustering & Annotation with scLucid

Clustering, resolution optimization, and cell type annotation — marker-based, CellTypist reference-based, and hybrid strategies.

## 1. Setup

In [ ]:
from scLucid.runtime import setup_runtime_environment
setup_runtime_environment()

import matplotlib
matplotlib.use('Agg', force=True)

import scanpy as sc
import pandas as pd
import scLucid

from scLucid.analysis import (
    ClusteringConfig,
    AnnotationConfig,
    ResolutionSearchConfig,
    cluster_cells,
    find_resolution,
    annotate_clusters,
    run_annotation,
    find_markers,
    run_enrichment,
)

sc.settings.set_figure_params(dpi=100, facecolor='white')

## 2. Load Preprocessed Data

In [ ]:
adata = sc.read_h5ad('data/pbmc3k.h5ad')
print(f'Cells: {adata.n_obs}, Genes: {adata.n_vars}')
if 'X_pca' not in adata.obsm:
    print('Running quick preprocessing ...')
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=50, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata)

## 3. Resolution Search

Find optimal clustering resolution by evaluating silhouette, marker abundance, and stability.

In [ ]:
resolution_config = ResolutionSearchConfig(
    method='leiden',
    use_rep='X_pca',
    resolution_range=(0.2, 2.0, 10),
    compute_silhouette=True,
    compute_marker_abundance=True,
    compute_stability=True,
    plot=True,
    save_dir='outputs/clustering',
)

eval_df, recommended = find_resolution(
    adata, config=resolution_config,
    auto_select=True, selection_strategy='balanced',
)
print(f'\nRecommended resolution: {recommended:.2f}')

## 4. Clustering

In [ ]:
adata = cluster_cells(adata, config=ClusteringConfig(
    method='leiden', resolution=recommended or 1.0,
    use_rep='X_pca', key_added='leiden_clusters', random_state=42,
))
print(f'Clusters: {adata.obs["leiden_clusters"].nunique()}')
sc.pl.umap(adata, color=['leiden_clusters'], legend_loc='on data')

## 5. Marker-Based Annotation

In [ ]:
adata = annotate_clusters(adata, config=AnnotationConfig(
    marker_species='human', marker_tissue='blood',
    cluster_key='leiden_clusters', marker_method='combined',
    key_added='cell_type_marker',
))
print(adata.obs['cell_type_marker'].value_counts().head(10))

## 6. Hybrid Annotation (Marker + CellTypist)

In [ ]:
try:
    result = run_annotation(adata, config=AnnotationConfig(
        cluster_key='leiden_clusters', marker_species='human',
        marker_tissue='blood', final_method='hybrid',
        run_celltypist=True, run_scoring=True, key_added='cell_type_hybrid',
    ))
    print(f'Hybrid labels: {adata.obs["cell_type_hybrid"].nunique()}')
except Exception as e:
    print(f'Hybrid annotation skipped: {e}')

## 7. Find Cluster Markers

In [ ]:
marker_df = find_markers(adata, groupby='leiden_clusters', method='wilcoxon', n_genes=20)
for cluster in sorted(marker_df['group'].unique())[:5]:
    top = marker_df[marker_df['group'] == cluster].head(3)
    print(f'\nCluster {cluster}:')
    for _, row in top.iterrows():
        print(f"  {row['names']:15s}  log2FC={row.get('logfoldchanges',0):.2f}")

## 8. Enrichment Analysis

In [ ]:
try:
    enrichment = run_enrichment(marker_df, method='gseapy', gene_sets='GO_Biological_Process_2023')
    if enrichment is not None:
        print(f'Enrichment done for {len(enrichment)} clusters')
except ImportError:
    print('Install gseapy for enrichment: pip install gseapy')

## 9. Save

In [ ]:
adata.write('outputs/annotated.h5ad')
print('Saved to outputs/annotated.h5ad')